# Détection automatique des Fake News grâce à l'IA

## Datasets

Les fichiers  et  ne sont pas inclus dans ce dépôt car ils sont trop volumineux.  
Téléchargez-les ici avant d'exécuter le notebook :  
👉 [https://www.kaggle.com/clmentbisaillon/fake-and-real-news-dataset](https://www.kaggle.com/clmentbisaillon/fake-and-real-news-dataset)

Une fois téléchargés, placez-les dans le **même dossier** que ce notebook.

In [ ]:
# =====================================================
# Détection automatique des Fake News grâce à l'IA
# VERSION COMPLÈTE POUR JUPYTERLITE / WASM
# =====================================================

# =====================================================
# INSTALLATION DES LIBRAIRIES
# EXÉCUTER UNE SEULE FOIS
# =====================================================

import micropip
await micropip.install(['pandas', 'numpy', 'scikit-learn', 'nltk', 'joblib'])

# =====================================================
# IMPORTATION DES LIBRAIRIES
# =====================================================

import pandas as pd
import numpy as np
import re
import nltk
import joblib
import ast
import operator as op

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# =====================================================
# TÉLÉCHARGEMENT DES DONNÉES NLTK
# =====================================================

nltk.download("stopwords")

# =====================================================
# CHARGEMENT DES DATASETS
# =====================================================

# IMPORTANT :
# Mettre Fake.csv et True.csv
# dans le même dossier que ce notebook

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

print("Taille du dataset Fake News :", fake.shape)
print("Taille du dataset Authentic News :", true.shape)

# =====================================================
# CRÉATION DES LABELS
# =====================================================

# 0 = Fake News
# 1 = Authentic News

fake["label"] = 0
true["label"] = 1

# =====================================================
# FUSION DES DATASETS
# =====================================================

news = pd.concat([fake, true], ignore_index=True)

# Mélange des lignes
news = news.sample(frac=1, random_state=42)

# Réinitialisation des index
news = news.reset_index(drop=True)

print("\nFusion des datasets réussie")

# =====================================================
# NETTOYAGE DES TEXTES
# =====================================================

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = str(text)

    # conversion en minuscules
    text = text.lower()

    # suppression des URLs
    text = re.sub(r"http\S+", "", text)

    # suppression des caractères spéciaux
    text = re.sub(r"[^a-zA-Z ]", "", text)

    # séparation des mots
    words = text.split()

    # suppression des stopwords
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Remplacement des valeurs manquantes
news["text"] = news["text"].fillna("")

# Nettoyage complet
news["text"] = news["text"].apply(clean_text)

print("Nettoyage des textes terminé")

# =====================================================
# SÉPARATION DES DONNÉES
# =====================================================

X = news["text"]
y = news["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTaille des données d'entraînement :", X_train.shape)
print("Taille des données de test :", X_test.shape)

# =====================================================
# TRANSFORMATION TF-IDF
# =====================================================

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTransformation TF-IDF terminée")

# =====================================================
# ENTRAÎNEMENT DU MODÈLE
# =====================================================

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("\nModèle entraîné avec succès")

# =====================================================
# ÉVALUATION DU MODÈLE
# =====================================================

predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("\n================================")
print("PRÉCISION DU MODÈLE :", accuracy)
print("================================")

print("\nRapport de classification :\n")
print(classification_report(y_test, predictions))

print("\nMatrice de confusion :\n")
print(confusion_matrix(y_test, predictions))

# =====================================================
# SAUVEGARDE DU MODÈLE
# =====================================================

joblib.dump(model, "fake_news_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\nModèle sauvegardé avec succès")

# =====================================================
# DÉTECTION DES EXPRESSIONS MATHÉMATIQUES
# =====================================================

def is_math_expression(text):

    math_symbols = ["+", "-", "*", "/", "=", ">", "<"]

    return any(symbol in text for symbol in math_symbols)

# =====================================================
# OPÉRATEURS AUTORISÉS
# =====================================================

operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg
}

# =====================================================
# ÉVALUATEUR MATHÉMATIQUE SÉCURISÉ
# =====================================================

def safe_eval(expr):

    def eval_node(node):

        if isinstance(node, ast.Num):
            return node.n

        elif isinstance(node, ast.BinOp):

            return operators[type(node.op)](
                eval_node(node.left),
                eval_node(node.right)
            )

        elif isinstance(node, ast.UnaryOp):

            return operators[type(node.op)](
                eval_node(node.operand)
            )

        else:
            raise TypeError(node)

    node = ast.parse(expr, mode="eval").body

    return eval_node(node)

# =====================================================
# VÉRIFICATION DES EXPRESSIONS MATHÉMATIQUES
# =====================================================

def check_math_expression(expression):

    try:

        allowed_chars = "0123456789+-*/().=<> "

        for char in expression:

            if char not in allowed_chars:
                return "FAKE NEWS"

        # ÉQUATION
        if "=" in expression and "==" not in expression:

            left, right = expression.split("=")

            left_value = safe_eval(left)
            right_value = safe_eval(right)

            if left_value == right_value:
                return "AUTHENTIC NEWS"

            else:
                return "FAKE NEWS"

        # SUPÉRIEUR
        elif ">" in expression:

            left, right = expression.split(">")

            if safe_eval(left) > safe_eval(right):
                return "AUTHENTIC NEWS"

            else:
                return "FAKE NEWS"

        # INFÉRIEUR
        elif "<" in expression:

            left, right = expression.split("<")

            if safe_eval(left) < safe_eval(right):
                return "AUTHENTIC NEWS"

            else:
                return "FAKE NEWS"

        # CALCUL NORMAL
        else:

            safe_eval(expression)

            return "AUTHENTIC NEWS"

    except Exception:

        return "FAKE NEWS"

# =====================================================
# FONCTION PRINCIPALE
# =====================================================

def predict_information(user_input):

    # Vérification mathématique
    if is_math_expression(user_input):

        return check_math_expression(user_input)

    # Détection Fake News

    cleaned_text = clean_text(user_input)

    vectorized_text = vectorizer.transform([cleaned_text])

    prediction = model.predict(vectorized_text)

    if prediction[0] == 1:
        return "AUTHENTIC NEWS"

    else:
        return "FAKE NEWS"

# =====================================================
# EXEMPLES DE TEST
# =====================================================

print("\n================================")
print("DÉTECTION AUTOMATIQUE DES FAKE NEWS GRÂCE À L'IA")
print("================================")

examples = [
    "2 + 2 = 4",
    "10 > 5",
    "5 * 7 = 20",
    "NASA discovered water on Mars"
]

for ex in examples:

    result = predict_information(ex)

    print("\nEntrée :", ex)
    print("Résultat :", result)

# =====================================================
# INTERACTION UTILISATEUR
# =====================================================

while True:

    user_text = input("\nEntrez une information (ou tapez exit) : ")

    if user_text.lower() == "exit":

        print("\nProgramme arrêté")
        break

    result = predict_information(user_text)

    print("\nRésultat :", result)